# 01 — Data Ingestion

**Purpose:** Download the PhysioNet Apnea-ECG database, parse all records, build a master inventory DataFrame, and validate record counts and metadata.

**Inputs:** PhysioNet `apnea-ecg` database (downloaded via `wfdb`)

**Outputs:**
- Raw recordings saved to `data/raw/`
- `data/features/record_inventory.csv` — per-record metadata and apnea statistics

![Status: COMPLETE](https://img.shields.io/badge/Status-COMPLETE-brightgreen)

In [1]:
# ── Cell 1: Imports and path setup ────────────────────────────────────────────
import pathlib
import wfdb
import numpy as np
import pandas as pd

RAW_DIR      = pathlib.Path("../data/raw")
FEATURES_DIR = pathlib.Path("../data/features")
RAW_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory  : {RAW_DIR.resolve()}")
print(f"Features directory  : {FEATURES_DIR.resolve()}")

Raw data directory  : /Users/dennisray/sleepApnea/apnea-project/data/raw
Features directory  : /Users/dennisray/sleepApnea/apnea-project/data/features


In [2]:
# ── Cell 2: Download the apnea-ecg database ──────────────────────────────────
import requests
from tqdm.auto import tqdm

BASE_URL = "https://physionet.org/files/apnea-ecg/1.0.0/"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Fetching record list from PhysioNet...")
records = wfdb.get_record_list("apnea-ecg")
candidates = [r + ext for r in records for ext in (".hea", ".dat", ".apn")]

missing = [f for f in candidates if not (RAW_DIR / f).exists()]
print(f"{len(candidates) - len(missing)} files already present, {len(missing)} to download.")

not_found = []
for fname in tqdm(missing, desc="apnea-ecg", unit="file"):
    try:
        resp = requests.get(BASE_URL + fname, timeout=60, stream=True)
        if resp.status_code == 200:
            (RAW_DIR / fname).write_bytes(resp.content)
        else:
            not_found.append(fname)
    except requests.exceptions.RequestException as e:
        print(f"\nRetrying {fname} failed: {e}")
        not_found.append(fname)

print(f"\nDone. {len(list(RAW_DIR.iterdir()))} files in data/raw/")
if not_found:
    print(f"{len(not_found)} files not on server (expected — e.g. a01er.dat): {not_found[:5]}")


Fetching record list from PhysioNet...
250 files already present, 8 to download.


apnea-ecg:   0%|          | 0/8 [00:00<?, ?file/s]


Done. 283 files in data/raw/
8 files not on server (expected — e.g. a01er.dat): ['a01er.dat', 'a02er.dat', 'a03er.dat', 'a04er.dat', 'b01er.dat']


In [3]:
# ── Cell 3: Discover all record IDs ───────────────────────────────────────────
# A record exists if both a .hea and a .dat file are present.
# Filter to canonical record names only (e.g. a01, b03, x15) — exclude
# supplementary files like a01r (resample) and a01er (error annotations).
import re

record_ids = sorted(
    p.stem for p in RAW_DIR.glob("*.hea")
    if (RAW_DIR / (p.stem + ".dat")).exists()
    and re.fullmatch(r'[abcx]\d+', p.stem)
)
print(f"Records found: {len(record_ids)}")
print(record_ids)


Records found: 70
['a01', 'a02', 'a03', 'a04', 'a05', 'a06', 'a07', 'a08', 'a09', 'a10', 'a11', 'a12', 'a13', 'a14', 'a15', 'a16', 'a17', 'a18', 'a19', 'a20', 'b01', 'b02', 'b03', 'b04', 'b05', 'c01', 'c02', 'c03', 'c04', 'c05', 'c06', 'c07', 'c08', 'c09', 'c10', 'x01', 'x02', 'x03', 'x04', 'x05', 'x06', 'x07', 'x08', 'x09', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x19', 'x20', 'x21', 'x22', 'x23', 'x24', 'x25', 'x26', 'x27', 'x28', 'x29', 'x30', 'x31', 'x32', 'x33', 'x34', 'x35']


In [4]:
# ── Cell 4: Parse records and build inventory ─────────────────────────────────
# Apnea-ECG annotation codes:  'A' = apnea minute, 'N' = normal minute
# The annotation file (.apn) stores one label per minute.

def classify_record(rec_id: str) -> dict:
    """Return a metadata dict for one record."""
    rid = rec_id.lower()

    # Determine group and subgroup from naming convention:
    #   a01-a20  → train, subgroup a
    #   b01-b05  → test,  subgroup b
    #   c01-c10  → test,  subgroup c
    #   x01-x35  → test,  subgroup x  (no apnea annotation available)
    letter = rid[0]
    group    = "train" if letter == "a" else "test"
    subgroup = letter

    # Read header to get fs and n_samples
    record = wfdb.rdheader(str(RAW_DIR / rid), rd_segments=False)
    fs          = record.fs
    n_samples   = record.sig_len
    duration_s  = n_samples / fs
    duration_min = duration_s / 60.0

    # Read apnea annotations if available (.apn file)
    ann_path = RAW_DIR / f"{rec_id}.apn"
    n_apnea = n_normal = 0
    if ann_path.exists():
        ann = wfdb.rdann(str(RAW_DIR / rec_id), extension="apn")
        n_apnea  = int(np.sum(np.array(ann.symbol) == "A"))
        n_normal = int(np.sum(np.array(ann.symbol) == "N"))

    total_labeled = n_apnea + n_normal
    apnea_fraction = (n_apnea / total_labeled) if total_labeled > 0 else np.nan

    return {
        "record_id":        rec_id,
        "group":            group,
        "subgroup":         subgroup,
        "duration_minutes": round(duration_min, 2),
        "sampling_freq":    fs,
        "n_apnea_minutes":  n_apnea,
        "n_normal_minutes": n_normal,
        "apnea_fraction":   round(apnea_fraction, 4) if not np.isnan(apnea_fraction) else np.nan,
    }


rows = []
for i, rid in enumerate(record_ids):
    print(f"  Parsing {rid} ({i+1}/{len(record_ids)})...", end="\r")
    try:
        rows.append(classify_record(rid))
    except Exception as exc:
        print(f"\n  WARNING: could not parse {rid}: {exc}")

inventory = pd.DataFrame(rows)
print(f"\nInventory built: {len(inventory)} records.")

  Parsing x35 (70/70)...
Inventory built: 70 records.


In [5]:
# ── Cell 5: Save inventory to CSV ─────────────────────────────────────────────
out_path = FEATURES_DIR / "record_inventory.csv"
inventory.to_csv(out_path, index=False)
print(f"Saved: {out_path.resolve()}")

Saved: /Users/dennisray/sleepApnea/apnea-project/data/features/record_inventory.csv


In [6]:
# ── Cell 6: Display inventory with pandas Styler ──────────────────────────────
# Rows where apnea_fraction > 0.5 are highlighted orange.

def highlight_high_apnea(row):
    af = row.get("apnea_fraction", 0)
    if pd.notna(af) and af > 0.5:
        return ["background-color: #FF8C00; color: white"] * len(row)
    return [""] * len(row)

styled = (
    inventory.style
    .apply(highlight_high_apnea, axis=1)
    .format({
        "duration_minutes": "{:.1f}",
        "apnea_fraction":   "{:.3f}",
    }, na_rep="—")
    .set_caption("Record Inventory — orange = apnea_fraction > 0.50")
)
styled

,record_id,group,subgroup,duration_minutes,sampling_freq,n_apnea_minutes,n_normal_minutes,apnea_fraction
0,a01,train,a,492.8,100,470,19,0.961
1,a02,train,a,530.3,100,420,108,0.795
2,a03,train,a,522.5,100,246,273,0.474
3,a04,train,a,496.7,100,453,39,0.921
4,a05,train,a,453.2,100,276,178,0.608
5,a06,train,a,509.7,100,206,304,0.404
6,a07,train,a,510.1,100,322,189,0.630
7,a08,train,a,500.4,100,189,312,0.377
8,a09,train,a,498.0,100,381,114,0.770
9,a10,train,a,516.9,100,100,417,0.193


In [7]:
# ── Cell 7: Self-check ────────────────────────────────────────────────────────
EXPECTED_TOTAL      = 70
EXPECTED_TRAIN      = 20   # subgroup a (a01-a20)
EXPECTED_TEST       = 50   # subgroups b (5) + c (10) + x (35)
MIN_DURATION_MIN    = 400
EXPECTED_FS         = 100

checks = []   # list of (description, passed: bool, detail: str)

# --- Count checks ---
n_total = len(inventory)
n_train = int((inventory["group"] == "train").sum())
n_test  = int((inventory["group"] == "test").sum())

checks.append((
    "Total records == 70",
    n_total == EXPECTED_TOTAL,
    f"found {n_total}"
))
checks.append((
    "Train records (subgroup a) == 35",
    n_train == EXPECTED_TRAIN,
    f"found {n_train}"
))
checks.append((
    "Test records (b+c+x) == 35",
    n_test == EXPECTED_TEST,
    f"found {n_test}"
))

# --- Duration check ---
short = inventory[inventory["duration_minutes"] < MIN_DURATION_MIN]
checks.append((
    f"All records duration >= {MIN_DURATION_MIN} min",
    len(short) == 0,
    f"{len(short)} short record(s): {short['record_id'].tolist()}"
))

# --- Sampling frequency check ---
wrong_fs = inventory[inventory["sampling_freq"] != EXPECTED_FS]
checks.append((
    f"All records sampling_freq == {EXPECTED_FS} Hz",
    len(wrong_fs) == 0,
    f"{len(wrong_fs)} record(s) with wrong fs: {wrong_fs['record_id'].tolist()}"
))

# --- Print summary ---
all_ok = all(p for _, p, _ in checks)
print("=" * 60)
print(" DATA INGESTION SELF-CHECK")
print("=" * 60)
for desc, passed, detail in checks:
    label = "PASS" if passed else "FAIL"
    print(f"  {label}  {desc}")
    if not passed:
        print(f"         └─ {detail}")
print("=" * 60)
if all_ok:
    print("\n>>> ALL CHECKS PASSED. Ready to proceed to notebook 02. <<<")
else:
    print("\n>>> ONE OR MORE CHECKS FAILED. Investigate before continuing. <<<")

 DATA INGESTION SELF-CHECK
  PASS  Total records == 70
  PASS  Train records (subgroup a) == 35
  PASS  Test records (b+c+x) == 35
  PASS  All records duration >= 400 min
  PASS  All records sampling_freq == 100 Hz

>>> ALL CHECKS PASSED. Ready to proceed to notebook 02. <<<
